In [1]:
!pip install yfinance
!pip install plotly

In [2]:
import numpy as np
import pandas as pd
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import yfinance as yf

In [3]:
ticker = "GOOG" # Googleの株価データを取得するためのティッカーシンボル
period = "1y"
interval = "1d"
data_df = yf.download(ticker, period=period, interval=interval)
data_df.columns = ['Open', 'High', 'Low', 'Close', 'Volume']
data_df

[*********************100%***********************]  1 of 1 completed


,Open,High,Low,Close,Volume
Date,,,,,
2025-03-14,167.042526,167.670361,163.943240,164.745475,18611100
2025-03-17,165.996124,167.879612,165.238733,166.748513,17839100
2025-03-18,162.109558,165.866574,158.252896,165.388232,24616800
2025-03-19,165.707123,167.553736,162.488255,163.350265,24955700
2025-03-20,164.481369,166.454543,162.577946,163.260583,19981500
...,...,...,...,...,...
2026-03-09,306.010010,306.500000,293.929993,294.135010,19632700
2026-03-10,306.929993,309.149994,305.309998,305.875000,14386400
2026-03-11,308.420013,311.069000,305.839996,306.299988,13376700


In [ ]:
fig = go.Figure( # グラフオブジェクトのFigureを作成し、
    data=[ # データとして、ローソク足チャートを追加
        go.Candlestick( # ローソク足チャートを作成するためのCandlestickオブジェクトを追加
            x=data_df.index,
            open=data_df["Open"],
            high=data_df["High"],
            low=data_df["Low"],
            close=data_df["Close"],
        )
    ]
)

fig.update_layout( # グラフのレイアウトを更新
    title="Google 株価",
    xaxis_title="日付",
    yaxis_title="株価（USD）"
)

fig.show()

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.2, subplot_titles=("ローソク足チャート", "出来高")) # 2行1列のサブプロットを作成し、x軸を共有し、縦のスペースを0.2に設定し、各サブプロットにタイトルを付ける

fig.add_trace(go.Candlestick( # ローソク足チャートを作成するためのCandlestickオブジェクトを追加
            x=data_df.index,
            open=data_df["Open"],
            high=data_df["High"],
            low=data_df["Low"],
            close=data_df["Close"],
            name="株価"
        ), row=1, col=1) # ローソク足チャートを1行目の1列目のサブプロットに追加

fig.add_trace(go.Bar( # 出来高を表す棒グラフを作成するためのBarオブジェクトを追加
    x=data_df.index,
    y=data_df["Volume"],
    marker_color="blue",
    name="出来高"
    ) ,row=2, col=1) # 出来高の棒グラフを2行目の1列目のサブプロットに追加


fig.update_layout(
    xaxis_rangeslider_visible=False,
    xaxis2_rangeslider_visible=True,
    xaxis2_title="日付",
    yaxis_title="株価",
    yaxis2_title="出来高",
)

fig.show()

## MACD

In [ ]:
short_ema = data_df["Close"].ewm(span=12, adjust=False).mean() 
# EMAは指数移動平均の略で、spanは期間を指定するパラメータで、adjust=Falseは過去のデータに対して指数的な重み付けを行うことを意味します。ここでは、12日間のEMAを計算しています。
long_ema = data_df["Close"].ewm(span=26, adjust=False).mean()
# 26日間のEMAを計算しています。一般的に、MACDは短期EMA（12日）と長期EMA（26日）の差を計算するため、これらの期間がよく使用されます。

macd = short_ema - long_ema

signal = macd.ewm(span=9, adjust=False).mean()

histogram = macd - signal

data_df["MACD"] = macd
data_df["Signal"] = signal
data_df["Histogram"] = histogram
data_df

,Open,High,Low,Close,Volume,MACD,Signal,Histogram
Date,,,,,,,,
2024-04-18,156.717712,157.737874,155.473605,156.185231,14016100,0.000000,0.000000,0.000000
2024-04-19,154.985916,157.245219,153.184451,157.006345,20063900,0.065502,0.013100,0.052402
2024-04-22,157.205399,158.434577,154.926201,155.274542,17243900,-0.022075,0.006065,-0.028140
2024-04-23,159.166122,159.723480,157.220337,157.842390,16115400,0.114405,0.027733,0.086672
2024-04-24,160.340546,160.629172,158.071295,158.340012,19485700,0.259726,0.074132,0.185594
...,...,...,...,...,...,...,...,...
2025-04-11,159.399994,159.860001,155.585007,155.585007,22582000,-5.856976,-6.131236,0.274260
2025-04-14,161.470001,164.029999,159.919998,162.309998,18255900,-4.884773,-5.881943,0.997171
2025-04-15,158.679993,162.050003,157.645004,161.570007,15690800,-4.126439,-5.530842,1.404404


In [9]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.2, subplot_titles=("ローソク足チャート", "MACD"))

fig.add_trace(go.Candlestick(
            x=data_df.index,
            open=data_df["Open"],
            high=data_df["High"],
            low=data_df["Low"],
            close=data_df["Close"],
            name="株価"
        ), row=1, col=1)

fig.add_trace(go.Scatter(x=data_df.index, y=data_df["MACD"], name="MACD", mode="lines"), row=2, col=1)
fig.add_trace(go.Scatter(x=data_df.index, y=data_df["Signal"], name="Signal", mode="lines"), row=2, col=1)
fig.add_trace(go.Bar(x=data_df.index, y=data_df["Histogram"], name="Histogram"), row=2, col=1)

fig.update_layout(
    xaxis_rangeslider_visible=False,
    xaxis2_rangeslider_visible=True,
    xaxis2_title="日付",
    yaxis_title="株価",
    yaxis2_title="MACD",
)

fig.show()